# Deep Learning for Corn Leaf Disease Classification

**Assignment:** Deep Learning for Image Classification & Segmentation (Image Analysis Group)

This notebook is a thin, reproducible orchestration layer over the project's modules
(`config.py`, `utils/`, `models/`, `train.py`, `evaluate.py`, `predict.py`). No logic is
duplicated here — every cell calls into the same code used by the command-line scripts and
the Streamlit UI (`app.py`), so results are guaranteed to be consistent across all three.

**Pipeline covered below:**
1. Dataset overview & class-imbalance / quality discussion
2. Preprocessing & augmentation visualization
3. Model architectures (baseline CNN vs. ResNet18 transfer learning)
4. Training both models (with checkpointing, early stopping, LR scheduling)
5. Evaluation on the held-out test set (metrics, confusion matrix, ROC/PR curves)
6. Baseline vs. advanced model comparison
7. Hyperparameter search demo
8. Single-image prediction demo
9. Conclusion

> Full 30-epoch training is time-consuming on CPU. For coursework submission, either run the
> `NOTEBOOK_EPOCHS` cell below with a small value for a quick, fully-reproducible demo, or run
> `python train.py --model both` from a terminal for the full run reported in the technical report.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebook" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import json
import pandas as pd
import matplotlib.pyplot as plt

import config
from utils.split_data import build_splits
from utils.dataset import CornLeafDataset, get_dataloaders
from utils.transforms import get_train_transforms, get_eval_transforms
from utils.visualize import (
    plot_class_distribution, plot_sample_images, plot_training_curves,
    plot_confusion_matrix, plot_roc_curve, plot_precision_recall_curve, plot_model_comparison,
)
from utils.metrics import count_parameters
from models import get_model

config.set_seed(config.SEED)
print(f"Device: {config.DEVICE}")
print(f"Classes ({config.NUM_CLASSES}): {config.CLASS_SHORT_NAMES}")

ModuleNotFoundError: No module named 'torchvision'

## 1. Dataset Overview

The raw data ships as `PlantVillage/train/<class>/*.jpg` and `PlantVillage/val/<class>/*.jpg`
(no test split). `utils/split_data.py` pools every image per class from both folders and
re-splits deterministically (seed=42) into train/validation/test manifests (70/15/15) stored
as CSVs under `dataset/splits/`. A manifest-based split was chosen over physically copying
images into `dataset/train|validation|test/` folders to avoid duplicating ~350MB of images on
disk while remaining exactly reproducible.

In [2]:
build_splits(force=False)

stats = json.loads((config.RESULTS_DIR / "dataset_stats.json").read_text())
print(f"Total images: {stats['total_images']}")
pd.DataFrame(stats["per_class"].items(), columns=["class", "count"])

2026-07-01 18:13:16 | INFO     | utils.split_data | Scanning raw dataset directories: [WindowsPath('C:/Users/warob/OneDrive/Desktop/DeepLearingAssignment/PlantVillage/train'), WindowsPath('C:/Users/warob/OneDrive/Desktop/DeepLearingAssignment/PlantVillage/val')]
2026-07-01 18:13:16 | INFO     | utils.split_data | Found 3852 images across 4 classes.
2026-07-01 18:13:16 | INFO     | utils.split_data | Wrote 2696 rows to C:\Users\warob\OneDrive\Desktop\DeepLearingAssignment\dataset\splits\train.csv
2026-07-01 18:13:17 | INFO     | utils.split_data | Wrote 578 rows to C:\Users\warob\OneDrive\Desktop\DeepLearingAssignment\dataset\splits\val.csv
2026-07-01 18:13:17 | INFO     | utils.split_data | Wrote 578 rows to C:\Users\warob\OneDrive\Desktop\DeepLearingAssignment\dataset\splits\test.csv
2026-07-01 18:13:17 | INFO     | utils.split_data | Dataset statistics saved to C:\Users\warob\OneDrive\Desktop\DeepLearingAssignment\results\dataset_stats.json
Total images: 3852


,class,count
0,Corn_(maize)___Cercospora_leaf_spot Gray_leaf_...,513
1,Corn_(maize)___Common_rust_,1192
2,Corn_(maize)___healthy,1162
3,Corn_(maize)___Northern_Leaf_Blight,985


In [ ]:
fig = plot_class_distribution(stats["per_class"], save_path=config.PLOTS_DIR / "class_distribution.png")
plt.show()

NameError: name 'plot_class_distribution' is not defined

### Dataset challenges

- **Class imbalance:** *Gray Leaf Spot* has roughly half the samples of *Common Rust* (the
  largest class). We counter this with a `WeightedRandomSampler` during training
  (`utils/dataset.py::get_dataloaders`) and inverse-frequency class weights in the loss
  (`utils/dataset.py::compute_class_weights`), rather than naively oversampling/duplicating files.
- **Visual similarity between diseases:** Northern Leaf Blight and Gray Leaf Spot both present
  as elongated necrotic lesions and are the most commonly confused pair in the confusion matrix.
- **Resolution/quality variance:** source images vary slightly in resolution and lighting;
  all images are resized to 224x224 and normalized with ImageNet statistics (required for the
  pretrained ResNet18 backbone, and applied uniformly to the baseline CNN for a fair comparison).
- **Background noise:** some leaf photos include soil, other leaves, or hands in frame, which
  augmentation (random crop/affine/perspective) helps the model become robust to.

## 2. Preprocessing & Augmentation

Training images go through `RandomResizedCrop`, `RandomHorizontalFlip`, `RandomRotation`,
`ColorJitter`, `RandomAffine`, and `RandomPerspective` (see `utils/transforms.py`).
Validation/test/inference use a deterministic `Resize` + `Normalize` pipeline only.

In [ ]:
preview_ds = CornLeafDataset(config.TRAIN_MANIFEST, transform=get_train_transforms())
fig = plot_sample_images(preview_ds, config.CLASS_NAMES, save_path=config.PLOTS_DIR / "sample_augmented_images.png", n=8)
plt.show()

## 3. Model Architectures

- **Baseline:** `CustomCNN` — 4 Conv-BatchNorm-ReLU-MaxPool blocks (3→32→64→128→256 channels),
  global average pooling, dropout, and a small FC classifier head. Trained from scratch.
- **Advanced:** `ResNet18` pretrained on ImageNet with the final FC layer replaced by
  `Dropout -> Linear(512, 4)`, fully fine-tuned end-to-end (transfer learning).

In [ ]:
baseline_model = get_model("baseline", num_classes=config.NUM_CLASSES)
resnet_model = get_model("resnet18", num_classes=config.NUM_CLASSES, pretrained=True)

print(f"CustomCNN trainable parameters: {count_parameters(baseline_model):,}")
print(f"ResNet18 trainable parameters:  {count_parameters(resnet_model):,}")
del baseline_model, resnet_model

## 4. Training

Set `NOTEBOOK_EPOCHS` below for a quick in-notebook demo run. This calls the exact same
`train_one_model` function used by `train.py`, so checkpoints/plots land in the same
`checkpoints/`, `plots/`, and `logs/` folders the CLI and the Streamlit app read from.

In [ ]:
import argparse
from train import train_one_model
from utils.logger import setup_logger

NOTEBOOK_EPOCHS = 5  # increase to 30 (or run `python train.py --model both`) for the reported results

notebook_args = argparse.Namespace(
    epochs=NOTEBOOK_EPOCHS, batch_size=config.DEFAULT_TRAINING_CONFIG.batch_size,
    lr=config.DEFAULT_TRAINING_CONFIG.learning_rate, weight_decay=config.DEFAULT_TRAINING_CONFIG.weight_decay,
    optimizer=config.DEFAULT_TRAINING_CONFIG.optimizer, patience=config.DEFAULT_TRAINING_CONFIG.early_stopping_patience,
    num_workers=0, seed=config.SEED, no_amp=(config.DEVICE.type != "cuda"), no_pretrained=False,
    no_class_weights=False, dropout=config.DEFAULT_TRAINING_CONFIG.dropout,
)
notebook_logger = setup_logger("notebook_train", config.LOGS_DIR / "notebook_train.log")

In [ ]:
baseline_meta = train_one_model("baseline", notebook_args, notebook_logger)
baseline_meta

In [ ]:
resnet_meta = train_one_model("resnet18", notebook_args, notebook_logger)
resnet_meta

In [ ]:
for name in ["baseline", "resnet18"]:
    history = json.loads((config.LOGS_DIR / f"{name}_history.json").read_text())
    plot_training_curves(history, model_name=name)
    plt.show()

## 5. Evaluation on the Test Set

Reuses `evaluate.py`'s `evaluate_model` function directly — metrics, confusion matrix, ROC
and precision-recall curves are written to `results/<model>/` and `plots/<model>/`.

In [ ]:
import argparse as _argparse
from evaluate import evaluate_model, build_comparison

eval_args = _argparse.Namespace(batch_size=32, num_workers=0)
eval_logger = setup_logger("notebook_eval", config.LOGS_DIR / "notebook_eval.log")

baseline_metrics = evaluate_model("baseline", config.best_ckpt_path("baseline"), eval_args, eval_logger)
resnet_metrics = evaluate_model("resnet18", config.best_ckpt_path("resnet18"), eval_args, eval_logger)

pd.DataFrame([{"model": "baseline", **baseline_metrics}, {"model": "resnet18", **resnet_metrics}])

In [ ]:
from PIL import Image as _Image

for name in ["baseline", "resnet18"]:
    for plot_file in ["confusion_matrix.png", "roc_curve.png", "precision_recall_curve.png"]:
        path = config.model_plots_dir(name) / plot_file
        if path.exists():
            display(_Image.open(path))

## 6. Baseline vs. Advanced Model Comparison

In [ ]:
build_comparison(["baseline", "resnet18"], eval_logger)
comparison_df = pd.read_csv(config.RESULTS_DIR / "model_comparison.csv")
comparison_df

In [ ]:
display(_Image.open(config.PLOTS_DIR / "model_comparison.png"))

## 7. Hyperparameter Search (demo)

`tune.py` performs a short grid/random search over learning rate, batch size, and optimizer
(`config.HP_SEARCH_SPACE`) using a reduced epoch budget, then ranks combinations by best
validation accuracy. Results are written to `results/hyperparam_search.csv`.

In [ ]:
# Uncomment to run the search inside the notebook (can be slow on CPU) —
# equivalent to: python tune.py --model baseline --epochs 3 --trials 4
#
# import sys as _sys
# _sys.argv = ["tune.py", "--model", "baseline", "--epochs", "3", "--trials", "4"]
# import tune
# tune.main()
# pd.read_csv(config.RESULTS_DIR / "hyperparam_search.csv")

## 8. Single-Image Prediction Demo

In [ ]:
import random
from predict import load_predictor, predict_image

test_df = pd.read_csv(config.TEST_MANIFEST)
sample_row = test_df.iloc[random.randrange(len(test_df))]
sample_path = Path(sample_row["filepath"])

predictor = load_predictor("resnet18", config.best_ckpt_path("resnet18"), config.DEVICE)
pred_class, confidence, probs, elapsed_ms = predict_image(predictor, sample_path, config.DEVICE)

print(f"True label:      {sample_row['label']}")
print(f"Predicted label: {pred_class}  (confidence={confidence:.2%}, {elapsed_ms:.1f} ms)")

display(_Image.open(sample_path))
pd.DataFrame({"class": config.CLASS_NAMES, "probability": probs}).sort_values("probability", ascending=False)

## 9. Conclusion

- The ResNet18 transfer-learning model is expected to outperform the from-scratch baseline CNN,
  especially in low-epoch regimes, since it starts from ImageNet-pretrained features.
- Class imbalance (Gray Leaf Spot underrepresented) is handled via weighted sampling + weighted
  loss rather than naive duplication.
- Most confusion is expected between Gray Leaf Spot and Northern Leaf Blight due to visually
  similar lesion patterns — see the per-model confusion matrices above.
- For the final report, re-run Section 4 with `NOTEBOOK_EPOCHS = 30` (or use
  `python train.py --model both`) to reproduce the full results, then re-run evaluation.
- An interactive Streamlit UI (`app.py`) is available for ad-hoc testing of trained checkpoints
  against new leaf photos: `streamlit run app.py`.